# Exploratory Data Analysis (EDA) on Congestion Dataset
This notebook loads `congestion_dataset_v3.csv` and performs basic exploratory data analysis, plotting data distributions, checking for class imbalance, and generating correlation matrices.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set display settings for better output
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

## 1. Data Loading & Basic Summary

In [ ]:
print("Loading dataset...")
df = pd.read_csv("congestion_dataset_v3.csv")

print(f"\nDataset Shape: {df.shape}")
print("\n--- First 5 rows ---")
display(df.head())

print("\n--- Dataset Info ---")
df.info()

## 2. Missing Values & Summary Statistics

In [ ]:
missing_values = df.isnull().sum()
print("Missing Values by Column:\n", missing_values[missing_values > 0])

print("\n--- Summary Statistics (Numeric Features) ---")
display(df.describe())

## 3. Class Distribution (Congestion)

In [ ]:
# Find congestion column
target_col = [c for c in df.columns if 'congest' in c.lower()]
if target_col:
    target_col = target_col[0]
    print(f"Found target column: {target_col}")
    
    plt.figure(figsize=(8, 6))
    sns.countplot(x=target_col, data=df)
    plt.title(f"Class Distribution of '{target_col}'")
    plt.xlabel("Congestion Status")
    plt.ylabel("Count")
    plt.show()
    
    print("Class Counts:\n", df[target_col].value_counts())
    print("\nClass Proportions:\n", df[target_col].value_counts(normalize=True))
else:
    print("No column containing 'congest' found.")

## 4. Feature Correlations

In [ ]:
# Separate numeric columns
numeric_df = df.select_dtypes(include=['float64', 'float32', 'int64', 'int32'])

if len(numeric_df.columns) > 1:
    plt.figure(figsize=(12, 10))
    corr = numeric_df.corr()
    # Generate a mask for the upper triangle
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap="coolwarm", vmax=1.0, vmin=-1.0, center=0,
                square=True, linewidths=.5, cbar_kws={"shrink": .5})
    plt.title("Correlation Matrix of Numeric Features")
    plt.show()
else:
    print("Not enough numeric features to plot correlation matrix.")

## 5. Feature Distributions vs Congestion

In [ ]:
if target_col and len(numeric_df.columns) > 1:
    # Select a few features to plot (exclude identifiers like ID/Index and the target itself)
    exclude = [target_col, 'Unnamed: 0', 'id', 'index']
    features_to_plot = [c for c in numeric_df.columns if c.lower() not in [e.lower() for e in exclude]][:6]
    
    if len(features_to_plot) > 0:
        fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(10, 4 * len(features_to_plot)))
        if len(features_to_plot) == 1:
            axes = [axes]
        
        for ax, feat in zip(axes, features_to_plot):
            sns.histplot(data=df, x=feat, hue=target_col, kde=True, bins=30, ax=ax, palette="Set2")
            ax.set_title(f"Distribution of {feat} by {target_col}")
            ax.set_xlabel(feat)
            ax.set_ylabel("Frequency")
            
        plt.tight_layout()
        plt.show()